# PCB Defect Detection — Colab Eğitim Defteri (CMPE 7144)
Akış: **GitHub'dan kodu çek → Drive'dan veriyi kopyala → ayrımı üret → `scripts/train.py` ile eğit → çıktılar Drive'da.**

Bu defter *ayar içermez*: epoch, görüntü boyutu, batch, tohum vb. hepsi depodaki `configs/train/*.yaml`
dosyasından okunur (yerelde de aynı dosya kullanılır). Burada yalnızca **hangi config** ve **run adı** seçilir.

Başlamadan önce:
1. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*
2. `DsPCBSD+.zip` dosyasını Drive'da `MyDrive/CMPE7144_PCB/data/` klasörüne yükle (~123 MB, bir kez).
3. Depo herkese açık; `GITHUB_TOKEN` gerekmez. (Özel yapılırsa Secrets'a eklenir, 4. hücre bunu otomatik kullanır.)


In [ ]:
!nvidia-smi

In [ ]:
# --- Seçimler (ayarların kendisi configs/train/*.yaml içinde) ---
GITHUB_USER  = "ethembelka"
REPO_NAME    = "pcb-surface-defect-detection"
DRIVE_ROOT   = "/content/drive/MyDrive/CMPE7144_PCB"
DATA_ZIP     = f"{DRIVE_ROOT}/data/DsPCBSD+.zip"
RUNS_DIR     = f"{DRIVE_ROOT}/runs"          # run klasörleri Drive'a yazılır; oturum kopsa da kalır
TRAIN_CONFIG = "configs/train/yolo11s_640.yaml"
RUN_ID       = "yolo11s_640_s0"              # sabit ad: --resume bunu bulur (tarihli otomatik ad kullanılmaz)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
# --- Kodu GitHub'dan çek, bağımlılıkları kur ---
import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
except Exception:
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"   # depo herkese açık
if os.path.exists(f"/content/{REPO_NAME}"):
    !git -C /content/{REPO_NAME} pull
else:
    !git clone {url} /content/{REPO_NAME}
%cd /content/{REPO_NAME}
!pip -q install -r requirements.txt
!git log --oneline -1

In [ ]:
# --- Veriyi Drive'dan yerel diske kopyala ve aç (Drive'dan doğrudan okumak çok yavaştır) ---
import glob
if not glob.glob("/content/DsPCBSD/**/Data_YOLO", recursive=True):
    !cp "{DATA_ZIP}" /content/dspcbsd.zip
    !unzip -q /content/dspcbsd.zip -d /content/DsPCBSD
yolo_root = glob.glob("/content/DsPCBSD/**/Data_YOLO", recursive=True)[0]
print("YOLO verisi:", yolo_root)

In [ ]:
# --- train_fit / val_internal listelerini üret (tohum sabit -> yereldekiyle birebir aynı ayrım) ---
!python scripts/make_splits.py --yolo-root "{yolo_root}"
!diff <(git show HEAD:data/splits/val_internal.txt) data/splits/val_internal.txt && echo "ayrım depodakiyle aynı"


In [ ]:
# --- Eğitim; yarım kalan run varsa kaldığı yerden devam eder ---
resume = "--resume" if os.path.exists(f"{RUNS_DIR}/{RUN_ID}/weights/last.pt") else ""
!python scripts/train.py --config {TRAIN_CONFIG} --data-root "{yolo_root}" --project "{RUNS_DIR}" --run-id {RUN_ID} {resume}
!cat "{RUNS_DIR}/{RUN_ID}/train_summary.json"

**Not:** Eğitim bitince resmi değerlendirme (COCO metrikleri, sınıf bazlı sonuçlar, karışıklık matrisi)
`scripts/evaluate.py` ile yapılır (aşama A3); Ultralytics'in kendi `val` çıktısı yalnızca hızlı bir sağlama içindir.